# 01 — Storm tracks and wind hazard (IBTrACS)

Finds every tropical cyclone that passed close enough, and hard enough, to
matter, and summarises each into one row of metrics.

**Run first.** Writes `outputs/ibtracs_impact_storms_{ISO3}_{start}_{end}.csv`,
read by notebooks `02` and `03`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Repo root on sys.path, whether launched from notebooks/ or the repo root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.patches import Rectangle
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cmcrameri import cm
from scipy.stats import binned_statistic_2d
from tabulate import tabulate

import config
from storm_utils import (
    load_boundary, load_ibtracs, add_distance, local_aeqd_crs,
    summarise_storms, impact_mask,
)
from plot_utils import normalise_cat, save_fig, set_style

config.ensure_dirs()
set_style()
print(config.summary())

### Input data

Download the IBTrACS v04r01 basin CSV from NOAA NCEI to `config.IBTRACS_CSV`:
<https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/>

Use the NCEI file, not the HDX mirror: HDX carries only `WMO_WIND`, which is
missing for many storm-hours.

## 2. Boundary and track data

In [ ]:
gdf, _ = load_boundary(config.ISO3)
CRS_METERS = local_aeqd_crs(gdf)
boundary = gdf.to_crs(CRS_METERS).union_all()
print(f"[OK] Boundary: {config.COUNTRY_NAME} ({config.ISO3})")

df = load_ibtracs(
    config.IBTRACS_CSV,
    basin=config.BASIN,
    start_year=config.START_YEAR,
    end_year=config.END_YEAR,
)
print(f"[OK] IBTrACS: {len(df):,} track points | {config.START_YEAR}–{config.END_YEAR}")
print("\n     Wind values by source agency:")
print(df["WIND_SOURCE"].value_counts().head().to_string())

In [ ]:
# Distance to the coastline, then the proximity filter
df = add_distance(df, boundary, CRS_METERS)
df["near_country"] = df["distance_km"] <= config.IMPACT_RADIUS_KM

nearby_sids = df.loc[df["near_country"], "SID"].unique()
nearby_track_points = df[df["SID"].isin(nearby_sids)].copy()

print(f"[OK] Track points within {config.IMPACT_RADIUS_KM:.0f} km of "
      f"{config.COUNTRY_NAME}: {df['near_country'].sum():,} / {len(df):,}")

## 3. The impact set

Two nested sets:

* **nearby storms** — came within `IMPACT_RADIUS_KM` at any intensity; used for
  the formation-zone maps.
* **trigger storms** — also met the intensity criteria in `config.IMPACT_CRITERIA`;
  the set the hazard statistics are built on.

In [ ]:
nearby_storms = summarise_storms(
    nearby_track_points, config.IMPACT_RADIUS_KM, config.ISO3, **config.IMPACT_CRITERIA
)
print(f"     Storms within {config.IMPACT_RADIUS_KM:.0f} km: {len(nearby_storms)}")

intensity_sids = {
    sid for sid, group in nearby_track_points.groupby("SID")
    if impact_mask(group, config.IMPACT_RADIUS_KM, **config.IMPACT_CRITERIA).any()
}
trigger_storms = nearby_storms[nearby_storms["sid"].isin(intensity_sids)].copy()
print(f"     Also meeting the intensity criteria: {len(trigger_storms)}")

trigger_storms.to_csv(config.STORMS_CSV, index=False)
print(f"  Saved → {config.STORMS_CSV}")

## 4. Summary statistics

In [ ]:
stats = {
    "Total storms"                        : len(trigger_storms),
    "Mean max intensity (km/h)"           : trigger_storms["max_wind_kmh"].mean(),
    "Median max intensity (km/h)"         : trigger_storms["max_wind_kmh"].median(),
    "Std dev max intensity (km/h)"        : trigger_storms["max_wind_kmh"].std(),
    "Mean travel days"                    : trigger_storms["travel_days"].mean(),
    "Median travel days"                  : trigger_storms["travel_days"].median(),
    "Std dev travel days"                 : trigger_storms["travel_days"].std(),
    "Mean closest approach (km)"          : trigger_storms["closest_dist_km"].mean(),
    "Min closest approach (km)"           : trigger_storms["closest_dist_km"].min(),
    "Mean distance at max intensity (km)" : trigger_storms["dist_at_max_intensity_km"].mean(),
    "Mean hours at hurricane force"       : trigger_storms["hours_as_hurricane"].mean(),
    # nullable boolean: True / False / None
    "Pct intensifying on approach"        : trigger_storms["intensifying_on_approach"]
                                            .astype("boolean").mean() * 100,
}

print(tabulate([(k, round(v, 2)) for k, v in stats.items()],
               headers=["Metric", "Value"], tablefmt="grid"))

In [ ]:
# Empirical return periods by category. Normalise first: ss_category is
# int / 'TS' in memory but a string after a CSV round-trip.
cat_labels = trigger_storms["ss_category"].apply(normalise_cat)
cat_counts = cat_labels.value_counts()
return_periods = {}

cat_1_2 = sum(cat_counts.get(c, 0) for c in ["Cat 1", "Cat 2"])
if cat_1_2:
    return_periods["Cat 1-2"] = round(config.RECORD_YEARS / cat_1_2, 1)

for c in ["Cat 3", "Cat 4", "Cat 5"]:
    if cat_counts.get(c, 0):
        return_periods[c] = round(config.RECORD_YEARS / cat_counts[c], 1)

hurricane_count = int(cat_labels.str.startswith("Cat ").sum())
if hurricane_count:
    return_periods["Any hurricane (Cat 1+)"] = round(config.RECORD_YEARS / hurricane_count, 1)
else:
    print("  [warn] no hurricane-force storms in the record")

order = ["Any hurricane (Cat 1+)", "Cat 1-2", "Cat 3", "Cat 4", "Cat 5"]
print(tabulate([(k, return_periods[k]) for k in order if k in return_periods],
               headers=["Category", "Return period (years)"], tablefmt="grid"))
print(f"\nRecord length: {config.RECORD_YEARS} years "
      f"({config.START_YEAR}–{config.END_YEAR})")

## 5. Distributions

> Return periods above are empirical — the observed rate over the record, with
> no distribution fitted and no confidence interval. The rarer categories rest
> on two or three storms.

In [ ]:
def plot_hist_kde(data, column, palette, xlabel, title, filename):
    """Histogram + KDE with mean and median lines."""
    fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
    sns.histplot(data[column], bins=20, kde=False, stat="density",
                 color=palette(0.7), alpha=0.8, ax=ax)
    sns.kdeplot(data[column], bw_adjust=0.5, color=palette(0.3), linewidth=2, ax=ax)
    ax.axvline(data[column].mean(), color="red", linestyle="--", linewidth=1.2,
               label=f"Mean: {data[column].mean():.1f}")
    ax.axvline(data[column].median(), color="green", linestyle="-", linewidth=1.2,
               label=f"Median: {data[column].median():.1f}")
    ax.set(title=title, xlabel=xlabel, ylabel="Density")
    ax.grid(False)
    ax.grid(axis="y", linewidth=0.3, alpha=0.4, color="grey")
    ax.tick_params(axis="x", length=2, width=0.5)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    sns.despine(ax=ax, left=True)
    ax.legend()
    plt.tight_layout()
    save_fig(fig, filename, config.OUTPUT_DIR)
    plt.show()


ISO3, CN = config.ISO3, config.COUNTRY_NAME

plot_hist_kde(trigger_storms, "max_wind_kmh", cm.batlow_r,
              "Max intensity (km/h)", f"Max intensity near {CN}",
              f"1a_intensity_distribution_{ISO3}.png")

plot_hist_kde(trigger_storms, "travel_days", cm.lajolla,
              "Travel days", f"Travel days from genesis — {CN}",
              f"1b_travel_days_{ISO3}.png")

plot_hist_kde(trigger_storms, "closest_dist_km", cm.batlow_r,
              "Closest approach distance (km)", f"Closest approach — {CN}",
              f"1c_closest_approach_{ISO3}.png")

## 6. Frequency over time

In [ ]:
years_index = range(config.START_YEAR, config.END_YEAR + 1)
all_nearby_counts = nearby_storms.groupby("year").size().reindex(years_index, fill_value=0)
trigger_counts = trigger_storms.groupby("year").size().reindex(years_index, fill_value=0)

rolling_all = all_nearby_counts.rolling(10, center=True, min_periods=5).mean()
rolling_impact = trigger_counts.rolling(10, center=True, min_periods=5).mean()

fig, ax = plt.subplots(figsize=(12, 5), dpi=150)
x = np.arange(config.START_YEAR, config.END_YEAR + 1)

ax.plot(x, rolling_all, color="#ABABFF", linewidth=1.8, linestyle="--",
        label="10-yr rolling mean (all nearby)", zorder=4)
ax.plot(x, rolling_impact, color="#8e44ad", linewidth=1.8, linestyle="--",
        label="10-yr rolling mean (impactful)", zorder=5)
ax.bar(x, all_nearby_counts.values, width=0.8, color="#fabba8", linewidth=0,
       label="Storm events (all nearby)", zorder=2)
ax.bar(x, trigger_counts.values, width=0.8, color="#ff5825", linewidth=0,
       label="Storm events (impactful)", zorder=3)

step = max(1, (config.END_YEAR - config.START_YEAR) // 12)
ax.set_xticks(x[::step])
ax.set_xticklabels(x[::step], rotation=45, ha="right")
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.set(
    xlabel="Year",
    ylabel="Number of tropical systems",
    title=f"Tropical systems near {CN} per year, {config.START_YEAR}–{config.END_YEAR}",
    xlim=(config.START_YEAR - 1, config.END_YEAR + 1),
    ylim=(0, all_nearby_counts.max() + 1),
)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(linewidth=0.3, alpha=0.3)
ax.legend(frameon=False)

plt.tight_layout()
save_fig(fig, f"1d_storm_events_by_year_{ISO3}.png", config.OUTPUT_DIR)
plt.show()

## 7. Formation-zone maps

Where the storms that reach this country form, and what they were doing on the
way. Each cell is the mean over storms whose genesis point falls in it.

> Counts per cell are small — read these as source regions, not per-cell statistics.

In [ ]:
centroid = gdf.to_crs("EPSG:4326").union_all().centroid
CENTR_LON, CENTR_LAT = centroid.x, centroid.y
HALF_LON, HALF_LAT = 40, 15          # degrees E/W, N/S around the country
EXTENT = [CENTR_LON - HALF_LON, CENTR_LON + HALF_LON,
          CENTR_LAT - HALF_LAT, CENTR_LAT + HALF_LAT]
GRID_SIZE = 0.55                     # degrees


def base_map(ax, title):
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, color="#f0ede8", zorder=1)
    ax.add_feature(cfeature.OCEAN, color="#daeaf5", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#aaaaaa", zorder=2)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":",
                   edgecolor="#aaaaaa", zorder=2)
    ax.plot(CENTR_LON, CENTR_LAT, marker="*", color="#000000", markersize=13, zorder=6)
    ax.set_title(title, fontsize=12, color="#3d3d3a", pad=10)
    for spine in ax.spines.values():
        spine.set_linewidth(0.3)
    gl = ax.gridlines(draw_labels=True, color="gray", alpha=0.2,
                      linestyle="--", linewidth=0.3)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER


def pixel_map(data, metric, cmap, label, title, filename,
              vmin=None, vmax=None, grid_size=GRID_SIZE, highlight=None):
    """Mean of `metric` binned by formation position; `highlight` outlines the
    cells where trigger storms formed."""
    lon_bins = np.arange(EXTENT[0], EXTENT[1] + grid_size, grid_size)
    lat_bins = np.arange(EXTENT[2], EXTENT[3] + grid_size, grid_size)

    d = data[data[["formation_lon", "formation_lat", metric]].notna().all(axis=1)]
    grid, _, _, _ = binned_statistic_2d(
        d["formation_lat"], d["formation_lon"], d[metric],
        statistic="mean", bins=[lat_bins, lon_bins],
    )

    fig, ax = plt.subplots(figsize=(14, 8), subplot_kw={"projection": ccrs.PlateCarree()})
    base_map(ax, title)
    mesh = ax.pcolormesh(lon_bins, lat_bins, grid, cmap=cmap,
                         transform=ccrs.PlateCarree(), zorder=3, vmin=vmin, vmax=vmax)

    if highlight is not None and len(highlight):
        h = highlight.dropna(subset=["formation_lon", "formation_lat"])
        counts, _, _, _ = binned_statistic_2d(
            h["formation_lat"], h["formation_lon"], h["formation_lat"],
            statistic="count", bins=[lat_bins, lon_bins],
        )
        for i, j in zip(*np.where(np.nan_to_num(counts) > 0)):
            ax.add_patch(Rectangle((lon_bins[j], lat_bins[i]), grid_size, grid_size,
                                   fill=False, edgecolor="#0000ff", linewidth=1.4,
                                   transform=ccrs.PlateCarree(), zorder=5))
        proxy = Rectangle((0, 0), 1, 1, fill=False, edgecolor="#0000ff", linewidth=1.4)
        ax.legend([proxy],
                  [f"Storms meeting the proximity and intensity criteria  n={len(h)}"],
                  loc="lower right", fontsize=7.5, frameon=False,
                  handlelength=1.2, handleheight=1.2, handletextpad=0.6).set_zorder(7)

    cb = plt.colorbar(mesh, ax=ax, pad=0.04, fraction=0.02, label=label)
    cb.outline.set_linewidth(0.3)
    cb.ax.tick_params(width=0.3, length=2)
    plt.tight_layout()
    save_fig(fig, filename, config.OUTPUT_DIR)
    plt.show()

In [ ]:
pixel_map(nearby_storms, "max_wind_kmh", cm.acton_r,
          "Mean peak intensity (km/h)",
          f"Peak intensity by formation zone — {CN}",
          f"1e_formation_peak_intensity_{ISO3}.png",
          highlight=trigger_storms)

pixel_map(nearby_storms, "formation_wind_kmh", "YlOrRd",
          "Wind speed at formation (km/h)",
          f"Wind speed at formation — {CN}",
          f"1f_formation_intensity_{ISO3}.png",
          highlight=trigger_storms)

pixel_map(nearby_storms, "travel_days", cm.buda_r,
          f"Mean travel days to {ISO3}",
          f"Travel days from formation — {CN}",
          f"1g_formation_travel_days_{ISO3}.png", vmin=0,
          highlight=trigger_storms)

pixel_map(nearby_storms, "formation_month", cm.romaO,
          "Mean formation month",
          f"Seasonality of formation zones — {CN}",
          f"1h_formation_seasonality_{ISO3}.png", vmin=1, vmax=12,
          highlight=trigger_storms)

---

**Next:** `02_rainfall_era5.ipynb` — ERA5-Land rainfall for each of these events.